# GRAG WebQSP training on Kaggle: two T4 GPUs

Consume a completed compact preprocessing backup from Google Drive; do not rerun embeddings or retrieval.
Train QLoRA adapters plus the original GAT and projector on `meta-llama/Llama-2-7b-chat-hf`.
The Llama backbone remains frozen in NF4 4-bit precision.

**Full run:** 10 total epochs, patience 4, microbatch 1, gradient accumulation 4.
**Text:** up to 512 description tokens, 64 generated answer tokens, 1,024 total positions.
Smoke mode uses one epoch on small subsets; switch it off only after verifying the complete workflow.

Two T4s are separate 16-GB devices, not one 32-GB device.
This notebook explicitly shards Llama layers across both GPUs in one process.
This is model parallelism, not DDP: it provides memory headroom but does not guarantee faster training or a fit for every graph.

Kaggle cannot use `google.colab.drive.mount`.
Drive API transfers provide verified persistence under the same logical layout as Colab:
`MyDrive/GRAG downstream/webqsp/<run-name>/`.
Local Drive-cache paths below are not a live filesystem mount.


## 1. Kaggle and account setup

1. Enable **Internet** and select **GPU T4 x2** in notebook settings.
2. Accept Meta's gated Llama-2 license on Hugging Face.
3. Add and enable these Kaggle Secrets for this notebook:
   - `HF_TOKEN`: Hugging Face read token with access to the gated model.
   - `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`: personal desktop OAuth credentials.
4. Copy the completed WebQSP snapshot folder ID from its Drive URL; it must contain `manifest.json` and all archive parts.
5. Copy the ID of your existing **GRAG downstream** folder in MyDrive; create that folder manually if absent.

**Drive permissions matter:** the preprocessing token's `drive.file` permission may see app-created backups
but not folders/checkpoints created by Colab. For those existing folders, explicitly reauthorize your desktop
OAuth application locally with `https://www.googleapis.com/auth/drive`, then store its refresh token in Kaggle Secrets.
This grants broad Drive access: do it only if acceptable to you. Full Drive scope is required here to avoid
creating duplicate folders when an app-limited token cannot see existing Colab-created children.
The notebook cannot expand a refresh token's grant: authentication fails if this scope was not authorized.
It never creates a replacement MyDrive root when access is missing.
Never paste tokens into cells, commit credentials, or publish secret-bearing output.

Push/upload the accompanying downstream helpers before cloning remotely.
Native Kaggle PyTorch/CUDA are retained; there is no author-era environment recreation.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 before running this notebook.')
for index in range(2):
    props = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {props.name}, {props.total_memory / 2**30:.2f} GiB')
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
ROOT = Path('/kaggle/working/BioGRAG-QA')
REPO_URL = 'https://github.com/MichaelPham2005/BioGRAG-QA.git'
REPO_BRANCH = 'GRAG/reproduce/WebQSP'
CODE_REVISION = ''  # Set the original Git commit when resuming after branch updates.
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
    if CODE_REVISION:
        subprocess.run(['git', '-C', str(ROOT), 'checkout', CODE_REVISION], check=True)
if not (ROOT / 'src/utils/notebook_downstream/drive_sync.py').is_file():
    raise FileNotFoundError('Updated Kaggle downstream helpers are missing; push/upload them first.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## 2. Install only missing dependencies

Do not replace Kaggle's PyTorch or CUDA installation.
If installation requires a Python restart, restart and rerun from section 1.


In [ ]:
requirements = {
    'torch_geometric': 'torch-geometric', 'transformers': 'transformers',
    'accelerate': 'accelerate', 'bitsandbytes': 'bitsandbytes', 'peft': 'peft',
    'sentencepiece': 'sentencepiece', 'pandas': 'pandas', 'tqdm': 'tqdm', 'requests': 'requests',
    'googleapiclient': 'google-api-python-client', 'google.auth': 'google-auth',
}
missing = []
for module, package in requirements.items():
    try:
        present = importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        present = False
    if not present:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    importlib.invalidate_caches()
from importlib.metadata import version
print({name: version(name) for name in requirements.values()})
from src.utils.notebook_downstream import artifacts, training
from src.utils.notebook_downstream.drive_sync import DriveFolder, restore_drive_snapshot
from src.utils.notebook_preprocessing.drive_backup import ensure_folder


## 3. Authenticate and select the experiment

Fill in the two folder IDs, not complete Drive URLs.
Use a new run name for this longer-context, two-GPU experiment; do not overwrite a Colab run.
Keep the same name/settings/code for resume. Never run two sessions writing the same Drive run folder.
Drive errors stop the notebook rather than continuing with local-only checkpoints.


In [ ]:
import re
from kaggle_secrets import UserSecretsClient
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google.auth.exceptions import RefreshError
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from huggingface_hub import hf_hub_download

SNAPSHOT_FOLDER_ID = ''  # Completed WebQSP preprocessing backup folder.
DOWNSTREAM_ROOT_ID = ''  # Your existing MyDrive/GRAG downstream folder.
for folder_id in (SNAPSHOT_FOLDER_ID, DOWNSTREAM_ROOT_ID):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', folder_id):
        raise ValueError('Fill in SNAPSHOT_FOLDER_ID and DOWNSTREAM_ROOT_ID with Drive folder IDs.')
SMOKE_TEST = True
DATASET = 'webqsp'
RUN_NAME = 'webqsp-chat-qlora-kaggle-2t4-' + ('smoke-v1' if SMOKE_TEST else 'full-v1')
MODEL_ID = 'meta-llama/Llama-2-7b-chat-hf'
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Enable HF_TOKEN in Kaggle Secrets and accept the Llama-2 license.')
hf_hub_download(MODEL_ID, 'config.json', token=HF_TOKEN)
credentials = Credentials(
    token=None, token_uri='https://oauth2.googleapis.com/token',
    client_id=secrets.get_secret('GDRIVE_CLIENT_ID'),
    client_secret=secrets.get_secret('GDRIVE_CLIENT_SECRET'),
    refresh_token=secrets.get_secret('GDRIVE_REFRESH_TOKEN'),
    scopes=['https://www.googleapis.com/auth/drive'],
)
try:
    credentials.refresh(Request())
except RefreshError:
    raise RuntimeError('Drive authentication failed. Check the client credentials and refresh token; '
                       'explicitly authorize full Drive scope locally for existing Colab folders.') from None
if (credentials.granted_scopes is not None
        and 'https://www.googleapis.com/auth/drive' not in credentials.granted_scopes):
    raise RuntimeError('Full Drive scope was not granted; stopping before creating any folders.')
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
try:
    root_info = service.files().get(fileId=DOWNSTREAM_ROOT_ID, fields='name,mimeType,capabilities(canAddChildren)').execute()
except HttpError as error:
    if error.resp.status not in (403, 404):
        raise
    raise RuntimeError('OAuth cannot access the existing downstream folder. Check its ID and explicitly '
                       'authorize Drive access as described above; no duplicate root was created.') from None
if (root_info['name'] != 'GRAG downstream'
        or root_info['mimeType'] != 'application/vnd.google-apps.folder'
        or not root_info.get('capabilities', {}).get('canAddChildren')):
    raise ValueError('DOWNSTREAM_ROOT_ID must identify your writable GRAG downstream folder.')
dataset_folder_id = ensure_folder(service, DATASET, DOWNSTREAM_ROOT_ID)
run_folder_id = ensure_folder(service, RUN_NAME, dataset_folder_id)
DRIVE_OUTPUT = Path('/kaggle/temp/drive/MyDrive/GRAG downstream') / DATASET / RUN_NAME
drive_store = DriveFolder(service, run_folder_id, DRIVE_OUTPUT)
DRIVE_OUTPUT = drive_store.cache
drive_store.restore_training_outputs()
OUTPUT = ROOT / 'output/downstream' / DATASET / RUN_NAME
artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)
saved_run = json.loads((OUTPUT / 'run.json').read_text()) if (OUTPUT / 'run.json').exists() else None
print('Drive path:', f'MyDrive/GRAG downstream/{DATASET}/{RUN_NAME}/')
print('Drive folder:', f'https://drive.google.com/drive/folders/{run_folder_id}')
print('Secrets are not written to run settings, checkpoints, or output files.')


## 4. Restore the verified preprocessing snapshot

Downloads and extraction are checksummed. Archive parts are removed from the temporary cache only after successful restore.
Prepared data and compact tensors remain local for lazy sample loading; model caches are outside Kaggle's saved working output.
Use the same snapshot when resuming. Smoke mode does not modify the source splits.


In [ ]:
PREPARED, PREPROCESS_RUN, SNAPSHOT_HASH = restore_drive_snapshot(
    service, SNAPSHOT_FOLDER_ID, Path('/kaggle/temp/grag-snapshots') / SNAPSHOT_FOLDER_ID,
    Path('/kaggle/temp/grag-restored') / SNAPSHOT_FOLDER_ID, DATASET,
)
dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
original_splits = dataset.get_idx_split()
limits = {'train': 8, 'val': 4, 'test': 4}
splits = {name: list(ids[:limits[name]] if SMOKE_TEST else ids) for name, ids in original_splits.items()}
print('Selected split counts:', {name: len(ids) for name, ids in splits.items()})
print('Excluded source examples:', dataset.metadata.get('excluded_count', 0))


## 5. Pin the model and shard it across two GPUs

Half the decoder layers run on each GPU. Embeddings, GAT and projector start on GPU 0;
the final normalization/output head run on GPU 1. Gradients flow across devices in one process.
Do not use `torchrun`, DDP, DataParallel, CPU/disk offloading, or Accelerate inference dispatch on top of this setup.

The explicit training settings below are fingerprinted. Ten epochs is an upper bound;
validation-loss early stopping uses patience 4. Changing settings requires a new run name.
Checkpoints can resume partial epochs and restore optimizer, AMP scaler and both GPUs' RNG states.
Keep two GPUs available when resuming this run.


In [ ]:
import random
from huggingface_hub import HfApi
from transformers import AutoConfig

requested_revision = saved_run['settings']['model_revision'] if saved_run else 'main'
model_revision = HfApi(token=HF_TOKEN).model_info(MODEL_ID, revision=requested_revision).sha
settings = training.Settings(
    dataset=DATASET, model_id=MODEL_ID, model_revision=model_revision,
    epochs=1 if SMOKE_TEST else 10, patience=4, batch_size=1, grad_accum=4,
    max_sequence_tokens=1024, max_description_tokens=512, max_answer_tokens=64,
    checkpoint_steps=50,
)
config = AutoConfig.from_pretrained(MODEL_ID, revision=model_revision, token=HF_TOKEN)
if config.model_type != 'llama' or config.tie_word_embeddings or config.num_hidden_layers < 2:
    raise ValueError('This two-GPU map expects Llama with untied embeddings and at least two decoder layers.')
if settings.max_sequence_tokens > config.max_position_embeddings:
    raise ValueError('The configured total budget exceeds the pinned model context limit.')
split_layer = config.num_hidden_layers // 2
device_map = {'model.embed_tokens': 0, 'model.norm': 1, 'lm_head': 1}
device_map.update({f'model.layers.{i}': 0 if i < split_layer else 1 for i in range(config.num_hidden_layers)})
SIGNATURE = training.initialize_run(
    settings, dataset, splits, SNAPSHOT_HASH, OUTPUT, DRIVE_OUTPUT,
    backup=drive_store.backup_file,
)
random.seed(settings.seed)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
model = training.load_qlora(settings, HF_TOKEN, device_map=device_map)
parameter_devices = {parameter.device for parameter in model.llm.parameters()}
if parameter_devices != {torch.device('cuda:0'), torch.device('cuda:1')}:
    raise RuntimeError(f'Expected parameters on both GPUs with no CPU/disk offload, got {parameter_devices}.')
print('Model revision:', model_revision, 'native context:', config.max_position_embeddings)
print('Trainable/total LLM parameters:', model.llm.get_nb_trainable_parameters())
print('Decoder layer split:', split_layer, '/', config.num_hidden_layers - split_layer)
runtime = {
    'code_commit': subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip(),
    'device_map': device_map, 'gpu_names': [torch.cuda.get_device_name(i) for i in range(2)],
    'packages': {name: version(name) for name in requirements.values()},
}
training.save_and_backup(runtime, 'runtime.json', OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)


## 6. Measure truncation before training

The description gets up to 512 tokens, but may be shortened to reserve the question, graph token,
delimiters, 64 answer tokens and EOS within 1,024 positions. Questions are never silently truncated.
Long WebQSP answer lists can exceed 64 tokens: inspect this report before committing to a full run.
Increasing a budget later changes the experiment fingerprint and requires a fresh run name.
These limits do not prune the GNN's soft graph.


In [ ]:
report = training.token_report(model, dataset, splits['train'] + splits['val'])
print('Training + validation token-limit report:', report)
training.save_and_backup(report, 'token-report.json', OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)


## 7. Train, validate and back up

Progress bars show optimizer updates, loss, learning rate, elapsed time and ETA.
Epoch summaries include validation and backup time. All checkpoint backups use verified Drive uploads:
`last.pt` every 50 optimizer steps and at epoch boundaries, `best.pt` when validation loss improves.

An upload failure raises an error and retains local files; fix access/network issues and rerun training to retry.
For a new Kaggle session, rerun sections 1-7 with the same run name/settings/code and two GPUs.
Do not resume from best.pt: last.pt carries optimizer and partial-epoch state.
A Kaggle timeout can lose updates since the last successful Drive backup.

For full training, set `SMOKE_TEST=False` in section 3 and rerun in a fresh session to release the smoke model.
The full run uses 10 epochs and patience 4; more epochs do not guarantee better test metrics.


In [ ]:
import time
for index in range(2):
    torch.cuda.reset_peak_memory_stats(index)
started = time.monotonic()
history = training.fit(
    model, dataset, splits, SIGNATURE, OUTPUT, DRIVE_OUTPUT,
    backup=drive_store.backup_file,
)
print(f'Training/resume elapsed: {(time.monotonic() - started) / 60:.1f} minutes')
print('Peak allocated GiB:', {i: round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(2)})


## 8. Evaluate the held-out test split

The best validation checkpoint is reloaded automatically, never selected by test scores.
Generation is greedy, with up to 64 new tokens. Raw predictions back up every 10 examples.
The prediction cache is bound to the exact best-checkpoint hash and selected test IDs.
Each best-checkpoint hash gets a separate evaluation subfolder, so continuation cannot reuse predictions from older weights.

WebQSP reports the source hit/coverage/F1/precision/recall metrics and retained/excluded counts.
Smoke results are labelled as subset results. Graph-only preprocessing and QLoRA are modified experiments,
not an exact reproduction of the paper's scores.


In [ ]:
evaluation_name = 'evaluation-best-' + artifacts.checksum(OUTPUT / 'best.pt')[:12]
evaluation_id = ensure_folder(service, evaluation_name, run_folder_id)
evaluation_store = DriveFolder(service, evaluation_id, DRIVE_OUTPUT / evaluation_name)
evaluation_store.restore_training_outputs()
EVALUATION_OUTPUT = OUTPUT / evaluation_name
artifacts.restore_training_outputs(evaluation_store.cache, EVALUATION_OUTPUT)
for name in ('run.json', 'best.pt'):
    artifacts.backup_file(OUTPUT / name, EVALUATION_OUTPUT / name)
training.save_and_backup(json.loads((OUTPUT / 'run.json').read_text()), 'run.json',
                         EVALUATION_OUTPUT, evaluation_store.cache, backup=evaluation_store.backup_file)
predictions, metrics = training.predict_test(
    model, dataset, splits['test'], SIGNATURE, EVALUATION_OUTPUT, evaluation_store.cache,
    backup=evaluation_store.backup_file,
)
print(json.dumps(metrics, indent=2))
for row in predictions[:3]:
    print('Question:', row['question'])
    print('Gold:', row['label'])
    print('Prediction:', row['prediction'])
    print('-' * 60)
print('Verified Drive evaluation output:', f'https://drive.google.com/drive/folders/{evaluation_id}')
